## Лабораторна робота 4
Брийовської Анастасії ОІ-34

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pyodbc
import os
import joblib


conn_str = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=Oleg;"
    "DATABASE=StudentStressMonitoring;"
    "Trusted_Connection=yes;"
)

conn = pyodbc.connect(conn_str)

df_Stress_Dataset = pd.read_sql(
    "SELECT * FROM dbo.Stress_DatasetClean", conn
)

df_StressLevelDataset = pd.read_sql(
    "SELECT * FROM dbo.StressLevelDatasetClean", conn
)

print("Stress_Dataset:", df_Stress_Dataset.shape)
print("StressLevelDataset:", df_StressLevelDataset.shape)

display(df_Stress_Dataset.head())
display(df_StressLevelDataset.head())

Stress_Dataset: (805, 29)
StressLevelDataset: (1101, 23)


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13612\4083734526.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_Stress_Dataset = pd.read_sql(
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13612\4083734526.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_StressLevelDataset = pd.read_sql(


,Gender,Age,Have you recently experienced stress in your life?,Have you noticed a rapid heartbeat or palpitations?,Have you been dealing with anxiety or tension recently?,Do you face any sleep problems or difficulties falling asleep?,Have you been dealing with anxiety or tension recently?.1,Have you been getting headaches more often than usual?,Do you get irritated easily?,Do you have trouble concentrating on your academic tasks?,...,Is your hostel or home environment causing you difficulties?,Do you lack confidence in your academic performance?,Do you lack confidence in your choice of academic subjects?,Academic and extracurricular activities conflicting for you?,Do you attend classes regularly?,Have you gained/lost weight?,Which type of stress do you primarily experience?,Distress (Negative Stress) - Stress that causes anxiety and impairs well-being.,Eustress (Positive Stress) - Stress that motivates and enhances performance.,No Stress - Currently experiencing minimal to no stress.
0,False,20,3,4,2,5,1,2,1,2,...,1,2,1,3,1,2,Eustress (Positive Stress) - Stress that motiv...,0,1,0
1,False,20,2,3,2,1,1,1,1,4,...,1,3,2,1,4,2,Eustress (Positive Stress) - Stress that motiv...,0,1,0
2,False,20,5,4,2,2,1,3,4,2,...,1,4,1,1,2,1,Eustress (Positive Stress) - Stress that motiv...,0,1,0
3,True,20,3,4,3,2,2,3,4,3,...,1,2,1,1,5,3,Eustress (Positive Stress) - Stress that motiv...,0,1,0
4,False,20,3,3,3,2,2,4,4,4,...,2,2,4,2,2,2,Eustress (Positive Stress) - Stress that motiv...,0,1,0


,anxiety_level,self_esteem,mental_health_history,depression,headache,blood_pressure,sleep_quality,breathing_problem,noise_level,living_conditions,...,study_load,teacher_student_relationship,future_career_concerns,social_support,peer_pressure,extracurricular_activities,bullying,stress_level,timestamp,ID
0,3.666667,3.666667,0,2.629630,2,1,2,4,2,3,...,2,3,3,2,3,3,2,1,2025-02-02 13:50:18,1
1,3.857143,2.066667,1,3.222222,5,3,1,4,3,1,...,4,1,5,1,4,5,5,2,2025-10-10 11:51:17,2
2,3.285714,3.400000,1,3.074074,2,1,2,2,2,2,...,3,3,2,2,3,2,2,1,2025-08-27 22:02:47,3
3,4.047619,2.600000,1,3.222222,4,3,1,3,4,2,...,4,1,4,1,4,4,5,2,2025-06-10 04:34:30,4
4,4.047619,4.733333,0,2.037037,2,3,5,1,3,2,...,3,1,2,1,5,0,5,1,2025-06-08 01:12:48,5


### Завдання 1. Створити сткуктуру моделі

Створено скруктуру моделі:

### CampusStressLevel

#### app/
1. database/
crud.py
database.py
models.py

2. endpoints/
inference.py
training.py

3. services/
model.py
utils.py

4. main.py
5. schemas.py

#### models/
#### src/
1. EDA.ipynb
2. Lab1.ipynb
3. Lab4.ipynb
#### README.md
#### requirement.txt

### Завдання 2. Підготувати базу даних

У базі даних StudentStressMonitoring створено дві додаткові таблиці:

- predictionsLab4 — призначена для збереження результатів прогнозування рівня стресу. Містить передбачений рівень стресу, джерело прогнозу (train або inference), унікальний ідентифікатор і дату створення запису.

- inference_inputs — призначена для збереження вхідних ознак, які надходять до моделі через HTTP-запит, а також ідентифікатора запису та дати його створення.

### Завдання 3. Реалізувати ендпоінт для тренування моделі.


У файлі training.py реалізовано ендпоінт для навчання моделі класифікації Decision Tree на основі даних із таблиці StressLevelDatasetClean

Основні етапи роботи:
1. Отримання всіх записів із бази даних та перетворення їх у DataFrame.
2. Видалення записів із відсутнім значенням цільової змінної stress_level.
3. Розділення даних на навчальну вибірку (90%) та нові вхідні дані (10%).
4. Заповнення пропущених значень середніми значеннями та масштабування ознак depression і anxiety_level у діапазон від 1 до 5.
5. Відокремлення вхідних ознак X_train від цільової змінної y_train.
6. Створення та навчання дерева рішень із використанням попередньо визначених найкращих гіперпараметрів.
7. Збереження навченої моделі та параметрів попередньої обробки у файлах за допомогою pickle
8. Отримання прогнозів на навчальних даних та збереження їх у таблиці predictions зі значенням source="train".


### Завдання 4. Реалізувати ендпоінт для передбачення.

У файлі реалізовано ендпоінт /predict, призначений для отримання прогнозів за допомогою раніше навченої моделі Decision Tree.

Основні етапи роботи:
1. Отримання даних із таблиці StressLevelDatasetClean та перетворення їх у DataFrame.
2. Видалення записів без цільової змінної stress_level та відокремлення 10% даних, які не використовувалися для навчання моделі.
3. Збереження початкових вхідних параметрів у форматі JSON.
4. Завантаження навченої моделі та параметрів попередньої обробки з файлів.
5. Заповнення пропущених значень середніми значеннями, отриманими під час навчання моделі.
6. Перевірка даних та масштабування необхідних колонок.
7. Виконання прогнозування за допомогою методу predict().
8. Збереження результатів прогнозування у таблиці predictions зі значенням source="inference", а початкових вхідних параметрів — у таблиці inference_inputs.

### Завдання 5. README.md та документація.

#### Campus Stress Level — FastAPI ML API

1. Запуск сервера FastAPI
2. Навчання моделі — POST /train-model
3. Передбачення — POST /predict
4. Структура бази даних
5. Приклади збережених передбачень

### Завдання 6. Опис проєкту.

### Логіка розділення даних

Для навчання та перевірки моделі використовується таблиця dbo.StressLevelDatasetClean у базі даних SQL Server. Спочатку записи з таблиці отримуються у порядку зростання ID. Рядки, у яких відсутнє значення цільової змінної stress_level, вилучаються, оскільки модель повинна навчатися на записах із відомим правильним значенням цільового класу.

Після цього дані поділяються на дві послідовні частини у співвідношенні 90:10:
1. перші 90% записів за ID використовуються для навчання моделі Decision Tree (DecisionTreeClassifier);
2. останні 10% записів за ID відкладаються для отримання передбачень за допомогою ендпоінта /predict.

Для однакового розділення в ендпоінтах /train-model та /predict використовується однакова послідовність дій: сортування за ID, вилучення записів без stress_level і визначення межі поділу split_index = int(len(df) * 0.9). Завдяки цьому обидва ендпоінти працюють з однаковими навчальною та відкладеною частинами.

Під час навчання пропущені значення окремих ознак заповнюються середніми значеннями, розрахованими на навчальній частині. Для вибраних ознак також виконується масштабування до діапазону від 1 до 5. Параметри попередньої обробки зберігаються у файлі models/param_dict.pickle і повторно використовуються під час передбачення. Відкладені 10% не використовуються для навчання моделі.

### Таблиці predictions і inference_inputs

#### Таблиця dbo.predictions

Зберігає результати роботи моделі. 
Вона містить такі колонки:
1. id - унікальний ідентифікатор запису, первинний ключ.
2. prediction - значення рівня стресу, передбачене моделлю.
3. timestamp - дата та час створення запису.
4. source - джерело передбачення (train для передбачень на навчальних даних або inference для передбачень на відкладених даних.)

Під час виконання /train-model результати передбачень на навчальних записах зберігаються зі значенням source = 'train'. Під час виконання /predict передбачення для останніх 10% записів зберігаються зі значенням source = 'inference'

#### Таблиця dbo.inference_inputs

Зберігає вхідні дані, для яких було виконано передбачення. 
Вона містить такі колонки:
Колонка	Призначення
id - унікальний ідентифікатор запису, первинний ключ.
input_data - вхідні параметри у форматі JSON.
timestamp - дата та час збереження вхідних параметрів.

У JSON-полі input_data зберігаються:

1. ID — ідентифікатор початкового запису з StressLevelDatasetClean;
2. 20 вхідних ознак: anxiety_level, self_esteem, mental_health_history, depression, headache, blood_pressure, sleep_quality, breathing_problem, noise_level, living_conditions, safety, basic_needs, academic_performance, study_load, teacher_student_relationship, future_career_concerns, social_support, peer_pressure, extracurricular_activities, bullying;
3. stress_level — фактичне значення цільової змінної з початкового датасету, яке зберігається для подальшого порівняння з передбаченням.

Фактичне значення stress_level зберігається для оцінювання результатів, але не передається моделі як вхідна ознака. Для передбачення використовуються лише 20 ознак із columns.X_columns.

### Подальший аналіз та заплановані інсайти

На основі таблиць predictions та inference_inputs планується виконувати такі дослідження:

1. Оцінювання якості моделі. Порівнювати передбачене значення prediction із фактичним stress_level для відкладених 10% даних (для цього зручно використовувати SQL-код з файлу README.md) та розраховувати Accuracy, Precision, Recall і Macro F1-score.
2. Аналіз помилок класифікації. Визначати, які класи рівня стресу модель розпізнає найкраще, а які найчастіше плутає.
3. Пошук закономірностей у помилкових прогнозах. Перевіряти, чи частіше модель помиляється за певних комбінацій вхідних характеристик.
4. Порівняння результатів навчання та передбачення. Оцінити, наскільки добре модель працює на даних, які не використовувалися під час її навчання.

За результатами одного з виконаних запусків модель правильно класифікувала 94 із 111 відкладених записів, що становить приблизно 84,68% Accuracy. Цей результат є попередньою оцінкою для конкретного запуску.